# 03 — Retrieval Evaluation (chunked corpus)

Identical to `02_evaluate_retrieval.ipynb` (same 10 questions, same four configs) but run against the **real chunked corpus** (`data/chunks.json`, ~325 chunks) instead of whole pages. Comparing the two notebooks' summary tables shows whether heading-aware chunking changes precision@1, and whether it narrows or widens the gap between OpenAI and local embeddings (see README's modality-gap discussion).

A question counts as a hit if **any** chunk belonging to the expected page is retrieved, matching the page-level granularity used in notebook 02.

Run `uv run hf-fetch` and `uv run hf-chunk` first. Needs `OPENAI_API_KEY` in `.env`.

In [1]:
import shutil
import tempfile
from pathlib import Path

import chromadb
import pandas as pd
from dotenv import load_dotenv

from hf_rag.config import make_minilm_embedding_fn, make_openai_embedding_fn
from hf_rag.eval.metrics import hit_rate
from hf_rag.retrieval.dense import DenseRetriever
from hf_rag.retrieval.reranking import RerankingRetriever

load_dotenv()
DATA = Path("../data")
tmp_dirs: list[str] = []

C:\Users\elvira\Desktop\data_science\hf_rag_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hand-picked questions (identical to notebook 02)

In [2]:
QUESTIONS = [
    {
        "query": "How do I load a model in 8-bit precision with bitsandbytes?",
        "expected_page": "quantization/bitsandbytes.md",
        "note": "Distractor: quantization/overview.md also mentions bitsandbytes in passing.",
    },
    {
        "query": "What's different about GPTQ quantization compared to other methods?",
        "expected_page": "quantization/gptq.md",
        "note": "Distractor: quantization/overview.md surveys all methods including GPTQ.",
    },
    {
        "query": "How do I apply LoRA adapters to a pretrained model with PEFT?",
        "expected_page": "peft.md",
        "note": "Distractor: training.md also references PEFT-based fine-tuning workflows.",
    },
    {
        "query": "Which Trainer arguments control the learning rate schedule and warmup?",
        "expected_page": "trainer.md",
        "note": "Distractor: training.md discusses TrainingArguments too, at a higher level.",
    },
    {
        "query": "How do I fine-tune a model to answer multiple-choice questions?",
        "expected_page": "tasks/multiple_choice.md",
        "note": "Distractor: tasks/question_answering.md is conceptually adjacent (also QA-flavored).",
    },
    {
        "query": "How do I prepare a dataset for object detection and compute mAP during evaluation?",
        "expected_page": "tasks/object_detection.md",
        "note": "Distractor: tasks/image_classification.md is the closest vision task guide.",
    },
    {
        "query": "How do I fine-tune a model for masked language modeling?",
        "expected_page": "tasks/masked_language_modeling.md",
        "note": "Distractor: tasks/causal_language_modeling.md is the other LM training objective.",
    },
    {
        "query": "How do I combine a model and processor for visual question answering?",
        "expected_page": "tasks/visual_question_answering.md",
        "note": "Distractor: tasks/image_classification.md also pairs a vision model with a processor.",
    },
    {
        "query": "How can I reduce GPU memory usage during training with gradient checkpointing or mixed precision?",
        "expected_page": "perf_train_gpu_one.md",
        "note": "Distractor: trainer.md exposes some of the same flags (fp16, gradient_checkpointing).",
    },
    {
        "query": "How do I fine-tune a translation model and evaluate it with BLEU?",
        "expected_page": "tasks/translation.md",
        "note": "Distractor: tasks/summarization.md is the other seq2seq generation task guide.",
    },
]

In [3]:
import json

with open(DATA / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

for q in QUESTIONS:
    q["relevant_ids"] = {
        c["chunk_id"] for c in chunks if c["page_path"] == q["expected_page"]
    }

print(f"{len(chunks)} chunks loaded, {len(QUESTIONS)} eval questions")

320 chunks loaded, 10 eval questions


## Build temporary collections (one document per chunk)

In [4]:
def build_temp_collection(
    label: str,
    ids: list[str],
    documents: list[str],
    metadatas: list[dict],
    embedding_fn,
):
    """Embed `documents` into a fresh on-disk Chroma collection so this notebook is self-contained
    and doesn't depend on what `hf-embed` has already persisted to data/chroma."""
    tmp_dir = tempfile.mkdtemp(prefix=f"hf_rag_eval_{label}_")
    tmp_dirs.append(tmp_dir)
    client = chromadb.PersistentClient(path=tmp_dir)
    collection = client.get_or_create_collection(
        name=label, metadata={"hnsw:space": "cosine"}
    )
    batch_size = 100
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i : i + batch_size]
        batch_docs = documents[i : i + batch_size]
        batch_metas = metadatas[i : i + batch_size]
        collection.upsert(
            ids=batch_ids,
            documents=batch_docs,
            embeddings=embedding_fn(batch_docs),
            metadatas=batch_metas,
        )
    return collection

In [5]:
chunk_ids = [c["chunk_id"] for c in chunks]
chunk_docs = [c["content"] for c in chunks]
chunk_metas = [
    {
        "page_path": c["page_path"],
        "page_title": c["page_title"],
        "heading": c["heading"],
    }
    for c in chunks
]

openai_embed_fn = make_openai_embedding_fn()
minilm_embed_fn = make_minilm_embedding_fn()

openai_collection = build_temp_collection(
    "openai_chunks", chunk_ids, chunk_docs, chunk_metas, openai_embed_fn
)
minilm_collection = build_temp_collection(
    "minilm_chunks", chunk_ids, chunk_docs, chunk_metas, minilm_embed_fn
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1981.85it/s]

## Evaluate: OpenAI vs. local, baseline vs. reranked

In [6]:
def run_evaluation(
    name: str, retriever, questions: list[dict], k_values=(1, 3, 5)
) -> dict:
    """Print a per-question OK/~/MISS line, return hit-rate@k averaged over `questions`."""
    print(f"\n=== {name} ===")
    max_k = max(k_values)
    hits = {k: [] for k in k_values}
    for item in questions:
        results = retriever.retrieve(item["query"], k=max_k)
        retrieved_ids = [r.chunk_id for r in results]
        relevant_ids = item["relevant_ids"]
        for k in k_values:
            hits[k].append(hit_rate(retrieved_ids[:k], relevant_ids))
        if hit_rate(retrieved_ids[:1], relevant_ids):
            status = "OK"
        elif hit_rate(retrieved_ids[:max_k], relevant_ids):
            status = "~"
        else:
            status = "MISS"
        print(f"  [{status:>4}] {item['query']}")
    return {f"hit@{k}": sum(v) / len(v) for k, v in hits.items()}

In [7]:
retrievers = {
    "OpenAI baseline": DenseRetriever(openai_collection, openai_embed_fn),
    "OpenAI + rerank": RerankingRetriever(
        DenseRetriever(openai_collection, openai_embed_fn)
    ),
    "Local baseline": DenseRetriever(minilm_collection, minilm_embed_fn),
    "Local + rerank": RerankingRetriever(
        DenseRetriever(minilm_collection, minilm_embed_fn)
    ),
}

summary = []
for name, retriever in retrievers.items():
    scores = run_evaluation(name, retriever, QUESTIONS)
    summary.append({"config": name, **scores})

pd.DataFrame(summary).set_index("config")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2062.49it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3601.23it/s]


=== OpenAI baseline ===


...

  [  OK] How do I fine-tune a model for masked language modeling?


  [  OK] How do I combine a model and processor for visual question answering?


  [MISS] How can I reduce GPU memory usage during training with gradient checkpointing or mixed precision?


  [  OK] How do I fine-tune a translation model and evaluate it with BLEU?


,hit@1,hit@3,hit@5
config,,,
OpenAI baseline,0.8,0.8,0.9
OpenAI + rerank,0.8,0.8,0.8
Local baseline,0.7,0.8,0.8
Local + rerank,0.8,0.8,0.8


## Compare against notebook 02

Re-run `02_evaluate_retrieval.ipynb` and place its summary table next to this one. The testable prediction (README, "two regimes"): chunking should help the local MiniLM embedder more than OpenAI's, since shorter, heading-anchored chunks reduce the burden on a smaller model's context window — so the OpenAI/local gap should narrow here relative to notebook 02.

## Cleanup

Temporary Chroma directories are scratch space for this notebook only; remove them so reruns start from a clean state.

In [8]:
for d in tmp_dirs:
    shutil.rmtree(d, ignore_errors=True)
print(f"Cleaned up {len(tmp_dirs)} temporary Chroma directories.")

Cleaned up 2 temporary Chroma directories.
